In [3]:
!pip install langchain langgraph -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_ollama import OllamaLLM

# jika Ollama serve default di localhost:11434, wrapper akan memakai itu
llm = OllamaLLM(model="llama3.1")  # ganti model sesuai yang sudah kamu pull

In [10]:
### LLM + LangGraph (3 NODE + CONDITIONAL ROUTING)

from typing import TypedDict
from langgraph.graph import StateGraph

class State(TypedDict):
    query: str
    answer: str

# --- NODE: router ---
def router(state: State):
    q = state["query"]
    if any(char.isdigit() for char in q):
        return {"next_node": "math_node"}      # → arahkan alurnya
    else:
        return {"next_node": "chat_node"}      # → arahkan alurnya

# --- NODE: math_node ---
def math_node(state: State):
    return {"answer": "Mohon maaf saya tidak bisa menjawab pesan berisi angka."}

# --- NODE: chat_node ---
def chat_node(state: State):
    q = state["query"]
    res = llm.invoke(q)
    return {"answer": res}

# --- BANGUN GRAPH ---
graph = StateGraph(State)

graph.add_node("router", router)
graph.add_node("math_node", math_node)
graph.add_node("chat_node", chat_node)

graph.set_entry_point("router")

# mapping edge berdasarkan nilai "route"
graph.add_conditional_edges(
    "router",
    lambda state: state["next_node"],
    {
        "math_node": "math_node",
        "chat_node": "chat_node"
    }
)

app = graph.compile()

output1 = app.invoke({"query": "Berapakah 26 - 11?"})   # → akan masuk math_node
print(output1)
output2 = app.invoke({"query": "Berapakah dua puluh enam dikurangi sebelas?"})   # → akan masuk chat_node
print(output2)
output3 = app.invoke({"query": "Bagaimana cara membayar UKT ITS?"})   # → akan masuk chat_node
print(output3)

{'query': 'Berapakah 26 - 11?', 'answer': 'Mohon maaf saya tidak bisa menjawab pesan berisi angka.'}
{'query': 'Berapakah dua puluh enam dikurangi sebelas?', 'answer': '26 – 11 = <<26-11=15>>15\nJawaban adalah 15.'}
{'query': 'Bagaimana cara membayar UKT ITS?', 'answer': 'Untuk informasi lebih lanjut, Anda dapat mengunjungi situs resmi ITS atau menghubungi tim layanan mahasiswa. Namun, secara umum, berikut adalah langkah-langkah yang biasanya diikuti dalam membayar UKT (Uang Kuliah Tunggal) di perguruan tinggi:\n\n1. **Pastikan Anda telah mendaftar sebagai mahasiswa**: Sebelum bisa membayar UKT, pastikan Anda telah berhasil menyelesaikan proses pendaftaran dan diterima sebagai mahasiswa.\n2. **Periksa ketersediaan biaya kuliah di semester tersebut**: Pastikan Anda telah memeriksa besarnya biaya kuliah yang harus dibayarkan untuk semester yang sedang berjalan atau akan datang. Jika ada, periksa juga ketersediaan bantuan dana kuliah jika dimungkinkan.\n3. **Siapkan rekening bank**: Pasti